<a href="https://colab.research.google.com/github/google-ai-edge/litert-samples/blob/main/benchmark/developer_device_platform/ddp_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2026 The AI Edge Authors.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ==============================================================================

# LiteRT / LiteRT-LM Benchmarking Using Developer Device Platform (DDP)


This notebook demonstrates how to benchmark edge models on real devices using [Developer Device Platform](https://docs.cloud.google.com/developer-device-platform/overview) (DDP), Google Cloud's managed device lab.

You can benchmark [LiteRT](https://github.com/google-ai-edge/litert) (`.tflite`) and [LiteRT-LM](https://github.com/google-ai-edge/LiteRT-LM) (`.litertlm`) models on a DDP device with a single `litert benchmark --ddp` command from the [LiteRT CLI](https://github.com/google-ai-edge/LiteRT-CLI).


## 🛠️ 1. Environment Setup & Installation

In [ ]:
# @title Setup your GCP project for DDP
# @markdown Set your GCP project id ([Developer Device Platform](https://docs.cloud.google.com/developer-device-platform/overview) is in Preview and its sessions are billed to this project, so it needs billing enabled; see the [quickstart](https://docs.cloud.google.com/developer-device-platform/quickstart)). The cell logs in to gcloud and enables the Device Run API.

ddp_gcp_project = "your-own-gcp-project-id" # @param {type:"string"}

if not ddp_gcp_project or ddp_gcp_project == "your-own-gcp-project-id":
  raise ValueError("Error: Please specify a valid GCP Project ID in the form above.")

# Login to gcloud and set Application Default Credentials via Colab Auth
from google.colab import auth
auth.authenticate_user()

# Enable the devicerun API
!gcloud services enable devicerun.googleapis.com --project {ddp_gcp_project}


## 🛠️ 2. Use LiteRT CLI to benchmark LiteRT / LiteRT-LM models

In [ ]:
# @title Install `LiteRT CLI`
# @markdown Installs the nightly build of the [LiteRT CLI](https://github.com/google-ai-edge/LiteRT-CLI), which ships the `--ddp` benchmark target.

# Install LiteRT CLI
!pip install litert-cli-nightly


In [ ]:
# @title Run `litert benchmark` to benchmark on DDP

# @markdown Set device type you want to benchmark. Device ids come from the DDP catalog, which you can
# @markdown list with `gcloud beta device-run devices list` (see the
# @markdown [DDP quickstart](https://docs.cloud.google.com/developer-device-platform/quickstart)).
ddp_device = "caiman-35" # @param {type:"string"}

# Download and benchmark litert models
# Download EfficientNet-B1 model directly from HuggingFace (No conversion needed)
!litert download litert-community/efficientnet_b1 --file "*.tflite" --output efficientnet

# Benchmark EfficientNet-B1 on a DDP device with CPU, then GPU
!litert benchmark efficientnet/efficientnet_b1.tflite --ddp --device {ddp_device} --cpu --gcp-project {ddp_gcp_project}
!litert benchmark efficientnet/efficientnet_b1.tflite --ddp --device {ddp_device} --gpu --gcp-project {ddp_gcp_project}

# Benchmark a LiteRT-LM bundle (prefill and decode tokens/s) on the same device's GPU
!litert download litert-community/Qwen3-0.6B --file qwen3_0_6b_mixed_int4.litertlm --output qwen3
!litert benchmark qwen3/qwen3_0_6b_mixed_int4.litertlm --ddp --device {ddp_device} --gpu --gcp-project {ddp_gcp_project}
